# Create regional extracts for county controls

## Introduction

This is a utility script intended to be used with the outputs from the county controls workflow, specifically the "All Data" table. It produces an Excel document for each of the counties in the MORPC region plus several county-based region definitions.  Each of the Excel documents contains several tabs containing wide-format tables and simple charts for the variables included in the All Data table.  As of October 2024, the county controls workflow is not well-documented, however some details are available in the [morpc-county-controls](https://github.com/morpc/morpc-county-controls) README.md file and the documents referenced therein.

This process is dependent on upstream processes. See the "Prerequisites" section below.

## Process outline

  - Open the "All Data" long-format table containing county controls variables for various geographies
  - Iterate through the geographies, producing an Excel document for each using the morpc.data_chart_to_excel() function

## Prerequisites and usage notes

  - Outputs of one or more upstream workflows must be available at the indicated paths. Make sure that those outputs are up to date prior to running this script. 
  - This script includes several intentional RuntimeError instances that may be triggered to alert the user to conditions that may require their attention. If the script triggers one of these errors, review the error, verify that the condition is acceptable or resolve any issues, then proceed.

## Setup

### Import required packages

In [1]:
import os
import shutil
import sys
import pandas as pd
import frictionless
sys.path.append(os.path.normpath("../morpc-common"))
import morpc

### User-specified parameters

In [2]:
# Several Excel documents containing extracts of the county controls variables for various counties/regions
# will be created in OUTPUT_DIR.  The directory will be created if necessary.
OUTPUT_DIR = "./deliverables/regional_extracts"

### Define inputs

The following datasets are required by this notebook.

#### MORPC counties reference data [81]

Reference data for counties in the MORPC region will be loaded automatically as a morpc.countyLookup() object (see below).

#### County Controls "All Data" table (CSV version) [256]

In [3]:
ALLDATA_RESOURCE_PATH = "./deliverables/CountyControls_WEB_AllData.resource.yaml"
print("Frictionless resource file: {}".format(ALLDATA_RESOURCE_PATH))

Frictionless resource file: ./deliverables/CountyControls_WEB_AllData.resource.yaml


### Define outputs

#### Create output data directory

Create output data directory if it doesn't exist.

In [4]:
outputDir = os.path.normpath(OUTPUT_DIR)
if not os.path.exists(outputDir):
    os.makedirs(outputDir)   

## Prepare input data

### Load county reference data

In [5]:
countyLookup = morpc.countyLookup(scope="15-County Region")

Loading data for MORPC 15-County region only


## Load county controls data

In [6]:
(allData, allDataResource, allDataSchema) = morpc.frictionless_load_data(ALLDATA_RESOURCE_PATH, validate=True)

morpc.load_frictionless_data | INFO | Loading Frictionless Resource file at location deliverables\CountyControls_WEB_AllData.resource.yaml
morpc.load_frictionless_data | INFO | Loading data, resource file, and schema from their source locations
morpc.load_frictionless_data | INFO | --> Data file: deliverables\CountyControls_WEB_AllData.csv
morpc.load_frictionless_data | INFO | --> Resource file: deliverables\CountyControls_WEB_AllData.resource.yaml
morpc.load_frictionless_data | INFO | --> Schema file: deliverables\CountyControls_WEB_AllData.schema.yaml
morpc.load_frictionless_data | INFO | Validating resource including data and schema.

Validating resource on disk (including data and schema). This may take some time.
Resource is valid
morpc.load_frictionless_data | INFO | Loading data.
Casting field COUNTY as type string.
Casting field CATEGORY as type string.
Casting field VARIABLE as type string.
Casting field 2020 as type integer.
Casting field 2021 as type integer.
Casting field 2

In [7]:
allData.head()

,COUNTY,CATEGORY,VARIABLE,2020,2021,2025,2030,2035,2040,2045,2050
0,15-County Region,Employment,Industrial Employment,271473,274455,286375,298045,314195,327874,340388,351959
1,10-County Region,Employment,Industrial Employment,247237,250283,262461,274305,290443,303964,316226,327549
2,CORPO Region,Employment,Industrial Employment,59045,59281,60221,62174,65130,68382,71687,74852
3,Columbus MSA,Employment,Industrial Employment,234481,237581,249968,261854,277735,291011,303107,314334
4,CEDS Region,Employment,Industrial Employment,255588,258604,270663,282445,298594,312195,324596,336048


## Transform data

### Configure output

In [8]:
regionName = "CEDS Region"
outputPath = os.path.join(outputDir, "County Controls - {}.xlsx".format(regionName))
print("Output file: {}".format(outputPath))

Output file: deliverables\regional_extracts\County Controls - CEDS Region.xlsx


In [9]:
try:
    del(excelWriter)
except:
    pass
excelWriter = pd.ExcelWriter(outputPath, engine='xlsxwriter')    

### Extract region data

In [10]:
allDataRegion = allData.loc[allData["COUNTY"] == regionName].copy() \
    .drop(columns=["COUNTY","CATEGORY"]) \
    .melt(id_vars="VARIABLE", var_name="YEAR", value_name="VALUE") \
    .rename(columns={"YEAR":"Year"}) \
    .pivot(index="Year", columns="VARIABLE", values="VALUE")

allDataRegion.head()

VARIABLE,Group Quarters Population,Household Population,Household Population 18 - 64,Household Population 65 and Older,Household Population Under 18,Households,Housing Units,Industrial Employment,Labor Force,Office Employment,...,Population - 80% Prediction Interval (-),Population - Confidence Limit (lower),Population - Confidence Limit (upper),Population Ages 18 - 64,Population Ages 65 and older,Population Ages under 18,Retail Goods Employment,Retail Services Employment,Total Employment,Workers
Year,,,,,,,,,,,,,,,,,,,,,
2020,61808,2189989,1353187,312319,524483,862783,934660,255588,1187770,501639,...,<NA>,<NA>,<NA>,1405320,320911,525566,112902,138249,1205895,1134681
2021,62119,2199564,1358266,313760,527538,866170,938246,258604,1198574,506075,...,<NA>,<NA>,<NA>,1410666,322388,528629,113826,139432,1216805,1145049
2025,64974,2319508,1403351,366202,549955,912706,988218,270663,1241792,523814,...,-103436,2281046,2420418,1458153,375222,551107,117519,144160,1260428,1186512
2030,67935,2437599,1453007,411735,572857,958107,1036891,282445,1292531,545040,...,-230751,2274783,2628330,1510311,421148,574075,122337,144921,1306228,1235148
2035,71562,2583348,1538169,444527,600652,1014325,1097262,298594,1365658,575748,...,-415196,2239714,2865039,1598533,454432,601945,129390,158922,1385320,1305472


### Detailed population forecast

In [11]:
sheetName = "Detailed population forecast"
columnMap = {
    "Population":"Forecasted population",
    "Population - Confidence Limit (lower)":"Confidence Limit (lower)",
    "Population - Confidence Limit (upper)":"Confidence Limit (upper)"
}

df = allDataRegion.copy() \
    .filter(items=columnMap.keys(), axis="columns") \
    .rename(columns=columnMap)

morpc.data_chart_to_excel(df, excelWriter, sheet_name=sheetName, chartType="line", 
    dataOptions={
        "numberFormat":{
            "Forecasted population": "#,##0",
            "Confidence Limit (lower)": "#,##0",
            "Confidence Limit (upper)": "#,##0"
        }
    },
    chartOptions={
        "colors": [morpc.CONST_MORPC_COLORS["darkgreen"], "#A6A6A6", "#A6A6A6"],                   # Specify a custom color
        "hideLegend": False,                             # Hide the legend
        "titles": {                                     # Specify the chart title and axis titles
            "chartTitle": "{} Population".format(regionName),
            "xTitle": None,
            "yTitle": None,
        }
    }
)

### Population

In [12]:
sheetName = "Population"
columnMap = {
    "Population":"Population"
}

df = allDataRegion.copy() \
    .filter(items=columnMap.keys(), axis="columns") \
    .rename(columns=columnMap)

baselineYear = df.index[0]
baselineValue = df.at[baselineYear, "Population"]

df["Change"] = (df["Population"] - baselineValue).astype("int")
df["Percent change"] = (df["Change"] / baselineValue).astype("float")
df["Population label"] = df["Population"].round(-2).apply(lambda x:"{:,}".format(x)).astype("string")
df["Percent change label"] = df["Percent change"].apply(lambda x:(("+{:.0%}".format(x)) if x>=0 else ("{:.0%}".format(x)))).astype("string")
df["Combined label"] = (df["Population label"] + "\n(" + df["Percent change label"] + ")").astype("string")

morpc.data_chart_to_excel(df, excelWriter, sheet_name=sheetName, chartType="column", 
    dataOptions={
        "numberFormat":{
            "Population": "#,##0",
            "Change": "#,##0",
            "Percent change": "#,##0.00",
            "Population label": "@",
            "Percent change label": "@",
            "Combined label": "@"
        }
    },
    chartOptions={
        "colors": [morpc.CONST_MORPC_COLORS["darkblue"]],                   # Specify a custom color
        "hideLegend": True,                             # Hide the legend
        "titles": {                                     # Specify the chart title and axis titles
            "chartTitle": "{0} forecasted population with percent change since {1}".format(regionName, baselineYear),
            "xTitle": None,
            "yTitle": None,
        },
        "includeColumns": ["Population"]
    }
)

### Households

In [13]:
sheetName = "Households"
columnMap = {
    "Households":"Households"
}

df = allDataRegion.copy() \
    .filter(items=columnMap.keys(), axis="columns") \
    .rename(columns=columnMap)

baselineYear = df.index[0]
baselineValue = df.at[baselineYear, "Households"]

df["Change"] = (df["Households"] - baselineValue).astype("int")
df["Percent change"] = (df["Change"] / baselineValue).astype("float")
df["Households label"] = df["Households"].round(-2).apply(lambda x:"{:,}".format(x)).astype("string")
df["Percent change label"] = df["Percent change"].apply(lambda x:(("+{:.0%}".format(x)) if x>=0 else ("{:.0%}".format(x)))).astype("string")
df["Combined label"] = (df["Households label"] + "\n(" + df["Percent change label"] + ")").astype("string")

morpc.data_chart_to_excel(df, excelWriter, sheet_name=sheetName, chartType="column", 
    dataOptions={
        "numberFormat":{
            "Households": "#,##0",
            "Change": "#,##0",
            "Percent change": "#,##0.00",
            "Households label": "@",
            "Percent change label": "@",
            "Combined label": "@"
        }
    },
    chartOptions={
        "colors": [morpc.CONST_MORPC_COLORS["darkgreen"]],
        "hideLegend": True,
        "titles": {
            "chartTitle": "{0} forecasted households with percent change since {1}".format(regionName, baselineYear),
            "xTitle": None,
            "yTitle": None,
        },
        "includeColumns": ["Households"]
    }
)

### Housing units

In [14]:
sheetName = "Housing units"
columnMap = {
    "Housing Units":"Housing units"
}

df = allDataRegion.copy() \
    .filter(items=columnMap.keys(), axis="columns") \
    .rename(columns=columnMap)

baselineYear = df.index[0]
baselineValue = df.at[baselineYear, "Housing units"]

df["Change"] = (df["Housing units"] - baselineValue).astype("int")
df["Percent change"] = (df["Change"] / baselineValue).astype("float")
df["Housing units label"] = df["Housing units"].round(-2).apply(lambda x:"{:,}".format(x)).astype("string")
df["Percent change label"] = df["Percent change"].apply(lambda x:(("+{:.0%}".format(x)) if x>=0 else ("{:.0%}".format(x)))).astype("string")
df["Combined label"] = (df["Housing units label"] + "\n(" + df["Percent change label"] + ")").astype("string")

morpc.data_chart_to_excel(df, excelWriter, sheet_name=sheetName, chartType="column", 
    dataOptions={
        "numberFormat":{
            "Housing units": "#,##0",
            "Change": "#,##0",
            "Percent change": "#,##0.00",
            "Housing units label": "@",
            "Percent change label": "@",
            "Combined label": "@"
        }
    },
    chartOptions={
        "colors": [morpc.CONST_MORPC_COLORS["midblue"]],
        "hideLegend": True,
        "titles": {
            "chartTitle": "{0} forecasted housing units with percent change since {1}".format(regionName, baselineYear),
            "xTitle": None,
            "yTitle": None,
        },
        "includeColumns": ["Housing units"]
    }
)

### Jobs

In [15]:
sheetName = "Jobs"
columnMap = {
    "Total Employment": "Total jobs",
    "Industrial Employment":"Industrial jobs",
    "Office Employment":"Office jobs",
    "Retail Goods Employment":"Retail goods jobs",
    "Retail Services Employment":"Retail services jobs",
    "Other Employment":"Other jobs"    
}

df = allDataRegion.copy() \
    .filter(items=columnMap.keys(), axis="columns") \
    .rename(columns=columnMap)

baselineYear = df.index[0]
baselineValue = df.at[baselineYear, "Total jobs"]

df["Total jobs change"] = (df["Total jobs"] - baselineValue).astype("int")
df["Percent change"] = (df["Total jobs change"] / baselineValue).astype("float")
df["Total jobs label"] = df["Total jobs"].round(-2).apply(lambda x:"{:,}".format(x)).astype("string")
df["Percent change label"] = df["Percent change"].apply(lambda x:(("+{:.0%}".format(x)) if x>=0 else ("{:.0%}".format(x)))).astype("string")
df["Combined label"] = (df["Total jobs label"] + "\n(" + df["Percent change label"] + ")").astype("string")

morpc.data_chart_to_excel(df, excelWriter, sheet_name=sheetName, chartType="column", 
    dataOptions={
        "numberFormat":{
            "Total jobs": "#,##0",
            "Office jobs": "#,##0",
            "Retail goods jobs": "#,##0",
            "Retail services jobs": "#,##0",
            "Industrial jobs": "#,##0",
            "Other jobs": "#,##0",            
            "Total jobs change": "#,##0",
            "Percent change": "#,##0.00",
            "Total jobs label": "@",
            "Percent change label": "@",
            "Combined label": "@"
        }
    },
    chartOptions={
        "subtype": "stacked",
        "colors": [
            morpc.CONST_MORPC_COLORS["darkgreen"],
            morpc.CONST_MORPC_COLORS["lightgreen"],
            morpc.CONST_MORPC_COLORS["darkblue"],
            morpc.CONST_MORPC_COLORS["bluegreen"],
            morpc.CONST_MORPC_COLORS["blue"]            
        ],
        "hideLegend": False,                             
        "titles": {                                     # Specify the chart title and axis titles
            "chartTitle": "{0} forecasted jobs by type with overall percent change since {1}".format(regionName, baselineYear),
            "xTitle": None,
            "yTitle": None,
        },
        "includeColumns": ["Office jobs","Retail goods jobs","Retail services jobs","Industrial jobs","Other jobs"]
    }
)

### Population by age

In [16]:
sheetName = "Population by age"
columnMap = {
    "Population": "Population",
    "Population Ages under 18": "Population ages 17 and under",
    "Population Ages 18 - 64": "Population ages 18 to 64",
    "Population Ages 65 and older":"Population ages 65 and older"
}

df = allDataRegion.copy() \
    .filter(items=columnMap.keys(), axis="columns") \
    .rename(columns=columnMap)

df["Population ages 17 and under (percent)"] = (df["Population ages 17 and under"] / df["Population"]).astype("float") 
df["Population ages 18 to 64 (percent)"] = (df["Population ages 18 to 64"] / df["Population"]).astype("float") 
df["Population ages 65 and older (percent)"] = (df["Population ages 65 and older"] / df["Population"]).astype("float") 

morpc.data_chart_to_excel(df, excelWriter, sheet_name=sheetName, chartType="column", 
    dataOptions={
        "numberFormat":{
            "Population": "#,##0",
            "Population ages 17 and under": "#,##0",
            "Population ages 18 to 64": "#,##0",
            "Population ages 65 and older": "#,##0",
            "Population ages 17 and under (percent)": "#,##0.00",
            "Population ages 18 to 64 (percent)": "#,##0.00",
            "Population ages 65 and older (percent)": "#,##0.00",
        }
    },
    chartOptions={
        "subtype": "stacked",
        "colors": [
            morpc.CONST_MORPC_COLORS["darkblue"],
            morpc.CONST_MORPC_COLORS["bluegreen"],
            morpc.CONST_MORPC_COLORS["blue"]            
        ],
        "hideLegend": False,                             
        "titles": {
            "chartTitle": "{0} forecasted population by age".format(regionName),
            "xTitle": None,
            "yTitle": None,
        },
        "includeColumns": ["Population ages 17 and under (percent)","Population ages 18 to 64 (percent)","Population ages 65 and older (percent)"]
    }
)

### Labor force (TBD)

### Close output

In [17]:
excelWriter.close()